In [1]:
import polars as pl
import pandas as pd
import numpy as np
import pyfaidx
import ast

In [5]:
from dna_utils import reverse_complement_dna, process_gtf, process_a_chrom

In [7]:
def process_a_file(data_file):
    variants = pd.read_csv(data_file)
    # Extract components from variant_id
    # Extract assembly from first variant_id (e.g. chr1_925969_C_T_hg38 -> hg38)
    assembly = variants['variant_id'].iloc[0].split('_')[-1]

    variants[['chrom', 'pos', 'ref', 'alt']] = variants['variant_id'].str.extract(r'(chr\d+|chrX|chrY)_(\d+)_([ACGT])_([ACGT])')
    variants['pos'] = variants['pos'].astype(int)
    variants = variants.sort_values(by=['chrom','pos']).reset_index(drop=True).reset_index()
    # Remove version numbers after dot in transcript_id
    variants['transcript_id'] = variants['transcript_id'].str.split('.').str[0]

    if assembly == 'hg19':  
        gtf_s, fasta = process_gtf('/data/reference/ucsc_gencodev27_hg19.tsv', fasta_path='/data/reference/hg19/hg19.fa')
    else:
        gtf_s, fasta = process_gtf('/data/reference/ucsc_gencodev32_hg38.tsv', fasta_path='/data/reference/hg38/hg38.fa')


    all_results = []
    chroms = variants['chrom'].unique()
    for chrom in chroms:
        curr_variants = variants[variants['chrom']==chrom]
        chrom_gtf = gtf_s[gtf_s['chrom']==chrom]
        chrom_results = process_a_chrom(curr_variants, chrom_gtf, return_alt_cds=True)
        all_results.append(chrom_results)
    all_results = pd.concat(all_results).reset_index(drop=True)
    all_results['variant_id'] = all_results['variant_id'] + '_' + assembly


    merged = variants.merge(all_results, left_on=['transcript_id','variant_id','chrom','pos','ref','alt'], 
                            right_on=['tx_name','variant_id','chrom','pos','ref','alt'], 
                        suffixes=('', '_y'))

    print(f"Original variants: {len(variants)}")
    print(f"After merging: {len(merged)}")
    missing = variants[~variants.variant_id.isin(merged.variant_id)]
    print("missing variants: ", len(missing))
    merged.to_csv(data_file.replace('.csv', '_processed.new.csv'), index=False)


Original variants: 2601
After merging: 2601
missing variants:  0


In [ ]:
data_files = ['/data/alphamissense_data/alphamissense_cancer_hotspot.csv',
              '/data/alphamissense_data/alphamissense_clinvar.csv',
              '/data/alphamissense_data/alphamissense_ddd.csv']
for data_file in data_files:
    process_a_file(data_file)